# 🔍 Bloque 0 — Auditoría de Calidad de Datos
**Objetivo:** Antes de analizar, entender qué tan confiables son los datos.

Para cada hallazgo documentamos:
- ✅ **Qué encontramos** (con números reales)
- ?� **Qué decidimos hacer** en los bloques siguientes

---

## ?� Setup: Cargar los datos

In [ ]:
import pandas as pd
import numpy as np

PATH = r"C:\Users\d0c00v5\Downloads\Datasets_extracted"

stores       = pd.read_csv(f"{PATH}/stores.csv")
products     = pd.read_csv(f"{PATH}/products.csv")
vendors      = pd.read_csv(f"{PATH}/vendors.csv")
promotions   = pd.read_csv(f"{PATH}/store_promotions.csv")
transactions = pd.read_csv(f"{PATH}/transactions.csv", parse_dates=["transaction_date"])
tx_items     = pd.read_csv(f"{PATH}/transaction_items.csv")

print(f"Transacciones: {len(transactions):,}")
print(f"Items:         {len(tx_items):,}")
print(f"Tiendas:       {len(stores):,}")
print(f"Productos:     {len(products):,}")
print(f"Proveedores:   {len(vendors):,}")

## ?� Explora cada tabla primero
Siempre explora antes de auditar. Corre estas celdas y observa.

In [ ]:
# Vista rápida de transactions
transactions.head(3)

In [ ]:
# Tipos de datos y valores nulos
transactions.info()

In [ ]:
# Estadísticas básicas
transactions.describe()

---
## 1️⃣ COMPLETITUD — ¿Faltan datos?
**Pregunta:** ¿Qué % de transacciones no tiene `customer_id`?

In [ ]:
total = len(transactions)

# --- customer_id ---
sin_cliente = transactions['customer_id'].isna().sum()
pct_sin     = sin_cliente / total * 100
print(f"Sin customer_id: {sin_cliente:,} de {total:,} ({pct_sin:.1f}%)")

# --- loyalty_card FALSE ---
loyalty_false = (transactions['loyalty_card'] == False).sum()
loyalty_true  = (transactions['loyalty_card'] == True).sum()
print(f"loyalty_card FALSE: {loyalty_false:,} ({loyalty_false/total*100:.1f}%)")
print(f"loyalty_card TRUE:  {loyalty_true:,} ({loyalty_true/total*100:.1f}%)")

# Son el mismo número? Tiene sentido: sin tarjeta = anónimo
print(f"\n¿Coinciden? {sin_cliente == loyalty_false}")

**?� Hallazgo:** ~60% de transacciones son anónimas. Es normal en retail.
**✅ Decisión:** Aceptar. Para análisis de cohortes usar solo `loyalty_card=TRUE`.

---
## 2️⃣ CONSISTENCIA — ¿Los números cuadran?
**Pregunta:** ¿El `total_amount` coincide con `SUM(unit_price * quantity)`?

In [ ]:
# Paso 1: suma de items por transacción
suma_items = (
    tx_items
    .assign(linea = tx_items['unit_price'] * tx_items['quantity'])
    .groupby('transaction_id')['linea']
    .sum()
    .reset_index()
    .rename(columns={'linea': 'suma_calculada'})
)

# Paso 2: comparar con total_amount
comparacion = transactions[['transaction_id','total_amount']].merge(
    suma_items, on='transaction_id'
)

# Paso 3: diferencia absoluta
comparacion['diferencia'] = (comparacion['total_amount'] - comparacion['suma_calculada']).abs()

# Paso 4: contar discrepantes (tolerancia $0.02 por redondeo)
discrepantes = (comparacion['diferencia'] > 0.02).sum()
print(f"Con discrepancia > $0.02: {discrepantes:,} ({discrepantes/total*100:.1f}%)")
print(f"Diferencia máxima: ${comparacion['diferencia'].max():.2f}")
print(f"Diferencia mediana: ${comparacion.loc[comparacion['diferencia']>0.02,'diferencia'].median():.2f}")

In [ ]:
# Ver las transacciones con mayor discrepancia
comparacion.nlargest(5, 'diferencia')

**?� Hallazgo:** 1% con discrepancia (redondeos/descuentos a nivel ticket).
**✅ Decisión:** Usar `total_amount` como fuente de verdad para GMV.

---
## 3️⃣ UNICIDAD — ¿Hay duplicados?
**Pregunta:** ¿Existen `transaction_id` repetidos?

In [ ]:
dup_tx    = transactions['transaction_id'].duplicated().sum()
dup_items = tx_items['transaction_item_id'].duplicated().sum()

print(f"Duplicados en transactions: {dup_tx}")
print(f"Duplicados en tx_items:     {dup_items}")

**?� Hallazgo:** 0 duplicados en ambas tablas. ✅
**✅ Decisión:** Sin acción requerida.

---
## 4️⃣ VALIDEZ — ¿Los valores tienen sentido?

In [ ]:
# total_amount negativos o cero
invalidos = (transactions['total_amount'] <= 0).sum()
print(f"total_amount <= 0: {invalidos} transacciones")

# Ver esas filas
transactions[transactions['total_amount'] <= 0]

In [ ]:
# unit_price = 0 sin estar en promo (sospechoso)
precio_0_sin_promo = tx_items[
    (tx_items['unit_price'] == 0) &
    (tx_items['was_on_promo'] == False)
].shape[0]

precio_0_con_promo = tx_items[
    (tx_items['unit_price'] == 0) &
    (tx_items['was_on_promo'] == True)
].shape[0]

print(f"unit_price=0 SIN promo (sospechoso): {precio_0_sin_promo}")
print(f"unit_price=0 CON promo (posible regalo): {precio_0_con_promo}")

**?� Hallazgo:** 3 transacciones con monto ≤0 | 231 items con precio $0 sin promo.
**✅ Decisión:** Excluir ambos del cálculo de GMV y GMROI.

---
## 5️⃣ INTEGRIDAD REFERENCIAL — ¿Todo está conectado?

In [ ]:
# store_id en transactions que NO existan en stores
tiendas_validas   = set(stores['store_id'])
tiendas_en_tx     = set(transactions['store_id'])
tiendas_huerfanas = tiendas_en_tx - tiendas_validas
print(f"Store IDs huérfanos: {tiendas_huerfanas}")

# vendor_id en products que NO existan en vendors
vendors_validos   = set(vendors['vendor_id'])
vendors_en_prod   = set(products['vendor_id'])
vendors_huerfanos = vendors_en_prod - vendors_validos
print(f"Vendor IDs huérfanos: {vendors_huerfanos}")

# Cuántos productos afecta?
prods_afectados = products[products['vendor_id'].isin(vendors_huerfanos)].shape[0]
print(f"Productos afectados: {prods_afectados}")

**?� Hallazgo:** 0 store_id huérfanos ✅ | VND_031 no existe en vendors (5 productos).
**✅ Decisión:** Excluir los 5 productos de VND_031 del análisis de GMROI.

---
## 6️⃣ FRESCURA — ¿Hay días sin ventas?

In [ ]:
transactions['fecha'] = transactions['transaction_date'].dt.date

# Fechas únicas por tienda
fechas_por_tienda = (
    transactions
    .groupby('store_id')['fecha']
    .apply(lambda x: sorted(x.unique()))
)

# Detectar gaps > 1 día
gaps = []
for tienda, fechas in fechas_por_tienda.items():
    for i in range(1, len(fechas)):
        gap = (fechas[i] - fechas[i-1]).days
        if gap > 3:  # gap real = gap - 1 dias sin ventas
            gaps.append({
                'tienda': tienda,
                'desde': fechas[i-1],
                'hasta': fechas[i],
                'dias_sin_ventas': gap - 1
            })

gaps_df = pd.DataFrame(gaps)
print(f"Gaps de más de 3 días detectados: {len(gaps_df)}")
gaps_df.sort_values('dias_sin_ventas', ascending=False)

**?� Hallazgo:** 1 gap de 7 días en 1 tienda. Puede ser inventario o festivo.
**✅ Decisión:** Investigar. Si es < 14 días, aceptar. Si > 14 días, excluir de Comp Sales.

---
## 7️⃣ INTEGRIDAD TEMPORAL — ¿Ventas antes de abrir?

In [ ]:
stores['opening_date'] = pd.to_datetime(stores['opening_date'])

# Join para comparar fechas
tx_con_apertura = transactions.merge(
    stores[['store_id', 'opening_date']],
    on='store_id'
)

# Transacciones ANTES de la apertura de la tienda
antes_apertura = tx_con_apertura[
    tx_con_apertura['transaction_date'] < tx_con_apertura['opening_date']
]

print(f"Transacciones antes de apertura: {len(antes_apertura)}")
print(f"Tiendas afectadas: {antes_apertura['store_id'].nunique()}")
antes_apertura[['transaction_id','store_id','transaction_date','opening_date']].head(5)

**?� Hallazgo:** 50 transacciones en 1 tienda con fecha antes de su apertura.
**✅ Decisión:** EXCLUIR. Son errores de carga.

---
## 8️⃣ INTEGRIDAD DEL A/B TEST — ¿Grupos contaminados?

In [ ]:
# ¿Alguna tienda está en CONTROL y TREATMENT a la vez?
variantes_por_tienda = (
    promotions
    .groupby(['store_id', 'promo_name'])['variant']
    .apply(set)
    .reset_index()
)

contaminadas = variantes_por_tienda[
    variantes_por_tienda['variant'].apply(
        lambda v: 'CONTROL' in v and 'TREATMENT' in v
    )
]

print(f"Tiendas en AMBOS grupos: {contaminadas['store_id'].tolist()}")
contaminadas

**?� Hallazgo:** TIENDA_008 y TIENDA_037 están en CONTROL y TREATMENT al mismo tiempo.
**✅ Decisión:** EXCLUIR del análisis A/B. Contaminan los resultados.

---
## ?� Resumen Final de Decisiones

| Dimensión | Hallazgo | Decisión |
|---|---|---|
| Completitud | 59.8% sin customer_id | Aceptar. Usar loyalty=TRUE para cohortes |
| Consistencia | 1% discrepancia total_amount | Usar total_amount como fuente de verdad |
| Unicidad | 0 duplicados | Sin acción |
| Validez | 3 negativos, 231 precio $0 | Excluir de GMV y GMROI |
| Integridad Ref. | VND_031 huérfano (5 prods) | Excluir del GMROI |
| Frescura | 1 gap de 7 días | Monitorear, aceptar si < 14 días |
| Integridad Temporal | 50 tx antes de apertura | Excluir |
| A/B Test | TIENDA_008 y 037 en ambos grupos | Excluir del experimento |
